# Commercial IP Downstream Evaluation — Formal Training Period

**Evaluation window:**
- In-time (train / val / test): `2024-11-20` → `2025-06-30` (~7 months)
- Out-of-time (OOT): `2025-07-01` → `2025-09-30`

**Notebook-ready modeling tables:**
| Role | BigQuery Table |
|---|---|
| New TE prejoined modeling table | `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Commercial_formal_training_full_downstream_new_te_features_outcomes_exp_round10_exp2b` |
| Production RAP prejoined modeling table | `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Commercial_formal_training_full_downstream_prod_rap_features_outcomes_exp_round10_exp2b` |

**Source lineage:**
| Role | BigQuery Table |
|---|---|
| Source New TE embeddings | `edp-prod-storage.edp_ent_sdoheir_cns.a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930` |
| Source Production RAP embeddings | `edp-prod-storage.edp_ent_sdoheir_cns.enhanced_rap_cp_emb_history_wide_4_te_fromal_eval` |
| Source tabular features + outcome | `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Commercial_final_dataset_4_te_formal_evaluation_20241120_20250930` |

**Attachment rule used in the prejoined tables:**
- New TE table: exact join on `individual_id` + `index_dt`
- Production RAP table: latest snapshot where `embedding.index_dt <= index_dt - 90 days`
- Preserve outcome `index_dt` as the business anchor date
- Store the matched embedding snapshot in `matched_embedding_index_dt` for audit only

**Why the join logic differs by source:**
- New TE embeddings are generated from the same formal-training cohort and share the same anchor date as the tabular outcome rows
- Production RAP embeddings behave like a dense history table with many dates per member, so they require an as-of lookup

**Split logic (same as baseline pipeline):**
- Train: `ind_id_last_digit` 0–7 AND `index_dt` ≤ cutoff
- Val: `ind_id_last_digit` 8 AND `index_dt` ≤ cutoff
- Test: `ind_id_last_digit` 9 AND `index_dt` ≤ cutoff
- OOT: `index_dt` > cutoff (all digits)
- OOT-strict: `index_dt` > cutoff AND `ind_id_last_digit` = 9

**Feature sets evaluated:** `embedding_only` | `tabular_only` | `hybrid`

**Comparisons:**
1. New TE prejoined table vs Production RAP prejoined table (embedding-only)
2. Which embedding source produces the stronger hybrid effect when combined with tabular features
3. SHAP feature importance for each hybrid model

## 1. Imports

In [ ]:
import gc
import os
import time
import warnings
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import pyarrow as pa
from tqdm.notebook import tqdm

import google.auth
from google.cloud import bigquery
from google.cloud.bigquery_storage import BigQueryReadClient, types

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.base import clone
from catboost import CatBoostClassifier, Pool
import shap

warnings.filterwarnings('ignore')

credentials, project = google.auth.default()
client = bigquery.Client()
bqstorage_client = BigQueryReadClient()
print(f'credentials: {credentials}')
print(f'project: {project}')

## 2. Constants

In [ ]:
# =============================================================================
# DATA SOURCES
# =============================================================================
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"

NEW_TE_MODELING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".a834793_Commercial_formal_training_full_downstream_new_te_features_outcomes_exp_round10_exp2b"
)
PROD_RAP_MODELING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".a834793_Commercial_formal_training_full_downstream_prod_rap_features_outcomes_exp_round10_exp2b"
)

# Lineage only: these raw-source tables are materialized into the modeling tables
# by data_ingestion/Formal_training_full_downstream/commercial/
# commercial_formal_training_full_downstream_embedding_features_outcomes.sql
# Mixed join semantics are intentional:
#   - New TE uses exact (individual_id, index_dt)
#   - Prod RAP uses latest embedding.index_dt <= outcome.index_dt - 90 days
SOURCE_NEW_TE_EMBEDDING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930"
)
SOURCE_PROD_RAP_EMBEDDING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".enhanced_rap_cp_emb_history_wide_4_te_fromal_eval"
)
SOURCE_TABULAR_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".a834793_Commercial_final_dataset_4_te_formal_evaluation_20241120_20250930"
)

MODELING_TABLES = {
    'new_te': NEW_TE_MODELING_TABLE,
    'prod_rap': PROD_RAP_MODELING_TABLE,
}

# =============================================================================
# SPLIT CONFIGURATION
# In-time:  2024-11-20 --> 2025-06-30
# OOT:      2025-07-01 --> 2025-09-30
# =============================================================================
OOT_CUTOFF_DATE = "2025-06-30"   # inclusive upper bound for in-time splits
FORMAL_EVAL_CUTOFF_DATE = OOT_CUTOFF_DATE

# =============================================================================
# TARGET & SPLIT KEY
# =============================================================================
TARGET_COLUMN = "ip6"
NEGATIVE_DOWNSAMPLE_RATIO = 10   # match baseline 10:1 negative sampling
GLOBAL_DOWNSAMPLE_RATIO = NEGATIVE_DOWNSAMPLE_RATIO

# =============================================================================
# COLUMNS TO EXCLUDE FROM FEATURES
# (identical to baseline pipeline — prevents leakage)
# =============================================================================
EXCLUDE_COLUMNS = frozenset([
    # Keys and identifiers
    'individual_id', 'member_id', 'index_dt', 'birth_dt', 'feature_end_dt',

    # Outcome columns
    'ip6', 'sum_ip6_admits', 'sum_ip6_los', 'sum_ip6_acu_days',

    # Eligibility / continuity flags
    'mon_3_include', 'mon_6_include', 'mon_12_include',
    'exclude_ip', 'include_post_6_status',

    # Split key
    'ind_id_last_digit',

    # Join audit column
    'matched_embedding_index_dt',

    # Leakage columns (cost amounts, outreach flags)
    'clm_allowed_amt_1yr', 'clm_allowed_amt_2yr', 'clm_allowed_amt_3mo', 'clm_allowed_amt_6mo',
    'clm_paid_amt_1yr', 'clm_paid_amt_2yr', 'clm_paid_amt_3mo', 'clm_paid_amt_6mo',
    'clm_par_allowed_amt_1yr', 'clm_par_allowed_amt_2yr', 'clm_par_allowed_amt_3mo', 'clm_par_allowed_amt_6mo',
    'clm_par_paid_amt_1yr', 'clm_par_paid_amt_2yr', 'clm_par_paid_amt_3mo', 'clm_par_paid_amt_6mo',
    'clm_srv_copay_amt_1yr', 'clm_srv_copay_amt_3mo', 'clm_srv_copay_amt_6mo',
    'covid_19', 'hpd_major_flag', 'chronic',
    'txt_member', 'txt_referral', 'txt_1yr_outreach', 'talked',
])

## 3. Metric Functions

In [ ]:
# =============================================================================
# METRIC FUNCTIONS
# =============================================================================

def lift_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Lift = precision@k / baseline_prevalence."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    precision_at_k = y_true[top_k_indices].mean()
    baseline = y_true.mean()
    return precision_at_k / baseline if baseline > 0 else 0.0


def true_positives_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> int:
    """Count true positives in top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return int(y_true[top_k_indices].sum())


def precision_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Precision (PPV) in top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return float(y_true[top_k_indices].mean())


def compute_split_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """Compute all metrics for a single split."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    return {
        'auc_roc':       roc_auc_score(y_true, y_prob),
        'auc_pr':        average_precision_score(y_true, y_prob),
        'brier':         brier_score_loss(y_true, y_prob),
        'lift_1pct':     lift_at_percentage(y_true, y_prob, 0.01),
        'lift_5pct':     lift_at_percentage(y_true, y_prob, 0.05),
        'lift_10pct':    lift_at_percentage(y_true, y_prob, 0.10),
        'tp_1pct':       true_positives_at_percentage(y_true, y_prob, 0.01),
        'precision_1pct': precision_at_percentage(y_true, y_prob, 0.01),
        'n_samples':     len(y_true),
        'n_positives':   int(y_true.sum()),
        'prevalence':    float(y_true.mean()),
    }

## 4. Data Preparation Functions

In [ ]:
# =============================================================================
# DATA PREPARATION FUNCTIONS
# =============================================================================

def downcast_numeric_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce numeric memory footprint without changing semantic meaning."""
    float64_cols = df.select_dtypes(include=['float64']).columns
    int64_cols = df.select_dtypes(include=['int64']).columns

    if len(float64_cols) > 0:
        df[float64_cols] = df[float64_cols].apply(pd.to_numeric, downcast='float')
    if len(int64_cols) > 0:
        df[int64_cols] = df[int64_cols].apply(pd.to_numeric, downcast='integer')
    return df


def _read_stream_to_arrow(
    stream_name: str,
    read_session: types.ReadSession,
) -> pa.Table:
    """Read one BigQuery Storage stream into a PyArrow Table (thread-safe)."""
    reader = bqstorage_client.read_rows(stream_name)
    return reader.to_arrow(read_session)


def load_bigquery_table_storage_api(
    table_id: str,
    max_stream_count: int = 10,
    optimize_dtypes: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Load a BigQuery table via the Storage Read API (gRPC + Arrow).

    Uses parallel streams for high-throughput reads (~50-100x faster than
    the REST-based list_rows API). Avoids the 403 "Response too large"
    error entirely since data is streamed directly from managed storage
    without materializing a query response.
    """
    table_ref = client.get_table(table_id)
    total_table_rows = table_ref.num_rows

    bq_table_path = f"projects/{table_ref.project}/datasets/{table_ref.dataset_id}/tables/{table_ref.table_id}"
    parent = f"projects/{table_ref.project}"

    requested_session = types.ReadSession(
        table=bq_table_path,
        data_format=types.DataFormat.ARROW,
    )

    session = bqstorage_client.create_read_session(
        parent=parent,
        read_session=requested_session,
        max_stream_count=max_stream_count,
    )

    n_streams = len(session.streams)
    if verbose:
        print(
            f"  Storage API session: {total_table_rows:,} rows, "
            f"{n_streams} parallel streams"
        )

    if n_streams == 0:
        return pd.DataFrame()

    arrow_tables = []
    with ThreadPoolExecutor(max_workers=n_streams) as executor:
        futures = {
            executor.submit(_read_stream_to_arrow, stream.name, session): i
            for i, stream in enumerate(session.streams)
        }

        pbar = tqdm(
            total=total_table_rows,
            desc=f"Loading {table_id.split('.')[-1][:45]}",
            unit=" rows",
            unit_scale=True,
            disable=not verbose,
        )

        for future in as_completed(futures):
            arrow_table = future.result()
            n_rows = arrow_table.num_rows
            arrow_tables.append(arrow_table)
            pbar.update(n_rows)
            pbar.set_postfix(
                streams_done=f"{len(arrow_tables)}/{n_streams}",
                remaining=f"{total_table_rows - pbar.n:,}",
            )

        pbar.close()

    combined = pa.concat_tables(arrow_tables)
    df = combined.to_pandas()
    del combined, arrow_tables
    gc.collect()

    if optimize_dtypes:
        df = downcast_numeric_dtypes(df)

    if verbose:
        mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
        print(
            f"  Loaded {len(df):,} rows, {len(df.columns)} columns, "
            f"{mem_mb:.0f} MB from {table_id}"
        )

    return df


def load_modeling_dataset(
    dataset_name: str,
    dataset_tables: Optional[Dict[str, str]] = None,
    max_stream_count: int = 10,
    verbose: bool = True,
    **kwargs,
) -> pd.DataFrame:
    """Load one modeling table via the BigQuery Storage Read API."""
    if dataset_tables is None:
        dataset_tables = MODELING_TABLES
    if dataset_name not in dataset_tables:
        raise KeyError(f"dataset_name not found in dataset_tables: {dataset_name}")

    table_id = dataset_tables[dataset_name]
    if verbose:
        print(f"Streaming modeling table: {dataset_name} -> {table_id}")
    return load_bigquery_table_storage_api(
        table_id,
        max_stream_count=max_stream_count,
        verbose=verbose,
    )


def query_modeling_table_summary(table_id: str) -> pd.DataFrame:
    """Run a lightweight aggregate query for one modeling table."""
    summary_sql = f"""
    SELECT
      COUNT(*) AS row_count,
      APPROX_COUNT_DISTINCT(CAST(individual_id AS STRING)) AS approx_unique_individuals,
      AVG(CAST({TARGET_COLUMN} AS FLOAT64)) AS prevalence,
      MIN(DATE(index_dt)) AS min_index_dt,
      MAX(DATE(index_dt)) AS max_index_dt,
      MIN(DATE(matched_embedding_index_dt)) AS min_matched_embedding_index_dt,
      MAX(DATE(matched_embedding_index_dt)) AS max_matched_embedding_index_dt
    FROM `{table_id}`
    """
    return client.query(summary_sql).to_dataframe()


def query_digit_distribution(table_id: str) -> pd.DataFrame:
    """Return split-key distribution without loading the full modeling table."""
    distribution_sql = f"""
    SELECT
      ind_id_last_digit,
      COUNT(*) AS row_count
    FROM `{table_id}`
    GROUP BY ind_id_last_digit
    ORDER BY ind_id_last_digit
    """
    return client.query(distribution_sql).to_dataframe()


def normalize_prejoined_modeling_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize a prejoined modeling table loaded from BigQuery.
    Ensures one row per (individual_id, index_dt) and keeps the matched
    embedding snapshot date only as an audit column.
    """
    df['index_dt'] = pd.to_datetime(df['index_dt']).dt.strftime('%Y-%m-%d')
    df['individual_id'] = df['individual_id'].astype(str)
    if 'matched_embedding_index_dt' in df.columns:
        df['matched_embedding_index_dt'] = pd.to_datetime(
            df['matched_embedding_index_dt']
        ).dt.strftime('%Y-%m-%d')
    df = df.drop_duplicates(subset=['individual_id', 'index_dt'], keep='last')
    return df


def create_data_splits(
    df: pd.DataFrame,
    oot_cutoff_date: str = OOT_CUTOFF_DATE,
) -> Dict[str, pd.DataFrame]:
    """
    Create train / val / test / OOT splits.

    Split logic (matches baseline pipeline):
        - Train:      ind_id_last_digit 0-7  AND  index_dt <= cutoff
        - Val:        ind_id_last_digit 8    AND  index_dt <= cutoff
        - Test:       ind_id_last_digit 9    AND  index_dt <= cutoff
        - OOT:        index_dt > cutoff  (all digits)
        - OOT-strict: index_dt > cutoff  AND  ind_id_last_digit 9
    """
    df = df.copy()
    df['_index_dt_parsed'] = pd.to_datetime(df['index_dt'])
    oot_cutoff = pd.to_datetime(oot_cutoff_date)

    splits = {
        'train': df[
            df['ind_id_last_digit'].isin([0, 1, 2, 3, 4, 5, 6, 7]) &
            (df['_index_dt_parsed'] <= oot_cutoff)
        ],
        'val': df[
            (df['ind_id_last_digit'] == 8) &
            (df['_index_dt_parsed'] <= oot_cutoff)
        ],
        'test': df[
            (df['ind_id_last_digit'] == 9) &
            (df['_index_dt_parsed'] <= oot_cutoff)
        ],
        'oot': df[df['_index_dt_parsed'] > oot_cutoff],
        'oot_strict': df[
            (df['_index_dt_parsed'] > oot_cutoff) &
            (df['ind_id_last_digit'] == 9)
        ],
    }

    print("Data splits created:")
    for name, split_df in splits.items():
        if len(split_df) > 0:
            prevalence = split_df[TARGET_COLUMN].mean() * 100
            print(f"  {name}: {len(split_df):,} rows, "
                  f"{int(split_df[TARGET_COLUMN].sum()):,} positives ({prevalence:.2f}%)")
        else:
            print(f"  {name}: EMPTY")

    for key in splits:
        splits[key] = splits[key].drop(columns=['_index_dt_parsed'])
    return splits


def identify_feature_columns(
    df: pd.DataFrame,
) -> Tuple[List[str], List[str]]:
    """
    Returns (embedding_features, tabular_features).
    Embedding features are columns starting with 'embedding_'.
    Tabular features are everything else after applying EXCLUDE_COLUMNS.
    """
    all_cols = set(df.columns)
    embedding_features = sorted([
        column_name for column_name in all_cols if column_name.startswith('embedding_')
    ])
    excluded = EXCLUDE_COLUMNS | set(embedding_features) | {'_exp_name', 'index_dt_parsed', '_index_dt_parsed'}
    tabular_features = sorted([
        column_name for column_name in all_cols
        if column_name not in excluded and column_name != TARGET_COLUMN
    ])
    return embedding_features, tabular_features


def downsample_negatives(
    X: pd.DataFrame,
    y: pd.Series,
    ratio: int = NEGATIVE_DOWNSAMPLE_RATIO,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Downsample the negative class to `ratio` negatives per positive.
    Matches baseline pipeline which used a pre-sampled table at 10:1.
    """
    np.random.seed(random_state)
    pos_indices = X.index[y == 1].tolist()
    neg_indices = X.index[y == 0].tolist()
    n_positives = len(pos_indices)
    n_negatives = len(neg_indices)
    target_n_negatives = int(n_positives * ratio)

    if n_negatives <= target_n_negatives:
        print(f"  Downsampling: no action needed (current ratio {n_negatives/n_positives:.1f}:1)")
        return X, y

    sampled_neg = np.random.choice(neg_indices, size=target_n_negatives, replace=False)
    keep = pos_indices + sampled_neg.tolist()
    X_res = X.loc[keep].copy()
    y_res = y.loc[keep].copy()
    shuffle_idx = np.random.permutation(len(X_res))
    X_res = X_res.iloc[shuffle_idx].reset_index(drop=True)
    y_res = y_res.iloc[shuffle_idx].reset_index(drop=True)
    print(f"  Downsampling: {n_negatives}:{n_positives} ({n_negatives/n_positives:.1f}:1) "
          f"→ {target_n_negatives}:{n_positives} ({ratio}:1)")
    return X_res, y_res


def prepare_features(
    df: pd.DataFrame,
    feature_cols: List[str],
) -> Tuple[pd.DataFrame, pd.Series]:
    """Build (X, y) for a given split, filling missing values."""
    X = df[feature_cols].copy()
    y = df[TARGET_COLUMN].astype(int)
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    X[numeric_cols] = X[numeric_cols].fillna(0)
    cat_cols = X.select_dtypes(include=['object', 'category']).columns
    X[cat_cols] = X[cat_cols].fillna('missing')
    return X, y

## 5. PreparedData Container & prepare_evaluation_data

In [ ]:
@dataclass
class PreparedData:
    """Container for prepared evaluation data. Prepare once, evaluate many models."""
    X_splits: Dict[str, pd.DataFrame]
    y_splits: Dict[str, pd.Series]
    feature_cols: List[str]
    embedding_features: List[str]
    tabular_features: List[str]
    cat_feature_indices: List[int]   # Column indices for CatBoost
    feature_set: str
    source_name: str
    downsampled: bool = True


def prepare_evaluation_data(
    df_features: pd.DataFrame,
    feature_set: str = 'embedding_only',
    oot_cutoff_date: str = OOT_CUTOFF_DATE,
    downsample_ratio: Optional[float] = None,
    random_state: int = 42,
    source_name: str = '',
) -> PreparedData:
    """
    Prepare one prejoined modeling table for multiple model evaluations.

    Args:
        df_features: Prejoined tabular + embedding + outcome DataFrame from BigQuery.
        feature_set: 'embedding_only' | 'tabular_only' | 'hybrid'
        oot_cutoff_date: Upper bound for in-time splits.
        downsample_ratio: Negative-to-positive ratio for training set rebalancing.
                          Use 10.0 to match baseline.
        random_state: Seed for downsampling.
        source_name: Short label describing the source table.

    Returns:
        PreparedData with X_splits, y_splits, and column metadata.
    """
    valid_feature_sets = {'embedding_only', 'tabular_only', 'hybrid'}
    if feature_set not in valid_feature_sets:
        raise ValueError(f"feature_set must be one of {valid_feature_sets}")

    total_start = time.time()

    # ------------------------------------------------------------------
    # Step 1: Normalize prejoined modeling table
    # ------------------------------------------------------------------
    print(
        f"\n[Step 1] Normalizing prejoined modeling table "
        f"(feature_set={feature_set}, source={source_name or 'unknown'})..."
    )
    step_start = time.time()
    df_prepared = normalize_prejoined_modeling_df(df_features)
    print(f"  Prepared shape: {df_prepared.shape}  ({time.time()-step_start:.1f}s)")

    # ------------------------------------------------------------------
    # Step 2: Create splits
    # ------------------------------------------------------------------
    print(f"\n[Step 2] Creating data splits (OOT cutoff: {oot_cutoff_date})...")
    splits = create_data_splits(df_prepared, oot_cutoff_date)

    # ------------------------------------------------------------------
    # Step 3: Identify feature columns
    # ------------------------------------------------------------------
    embedding_features, tabular_features = identify_feature_columns(df_prepared)
    if feature_set == 'embedding_only':
        feature_cols = embedding_features
    elif feature_set == 'tabular_only':
        feature_cols = tabular_features
    else:  # hybrid
        feature_cols = tabular_features + embedding_features

    print(f"  {len(feature_cols)} features selected "
          f"({len(embedding_features)} embedding, {len(tabular_features)} tabular)")

    # ------------------------------------------------------------------
    # Step 4: Build (X, y) for each split
    # ------------------------------------------------------------------
    print("\n[Step 4] Preparing feature matrices...")
    X_splits, y_splits = {}, {}
    for split_name, split_df in splits.items():
        if len(split_df) > 0:
            X_splits[split_name], y_splits[split_name] = prepare_features(split_df, feature_cols)

    # ------------------------------------------------------------------
    # Step 5: Downsample negatives in training set
    # ------------------------------------------------------------------
    downsampled = False
    if downsample_ratio is not None and 'train' in X_splits:
        print(f"\n[Step 5] Downsampling training set to {downsample_ratio}:1 ratio...")
        X_splits['train'], y_splits['train'] = downsample_negatives(
            X_splits['train'], y_splits['train'],
            ratio=downsample_ratio, random_state=random_state,
        )
        downsampled = True

    # Pre-compute categorical column indices for CatBoost
    cat_feature_indices = []
    if feature_set != 'embedding_only' and 'train' in X_splits:
        cat_cols = X_splits['train'].select_dtypes(include=['object', 'category']).columns
        cat_feature_indices = [X_splits['train'].columns.get_loc(c) for c in cat_cols]

    print(f"\nData preparation complete ({time.time()-total_start:.1f}s total)")
    return PreparedData(
        X_splits=X_splits,
        y_splits=y_splits,
        feature_cols=feature_cols,
        embedding_features=embedding_features,
        tabular_features=tabular_features,
        cat_feature_indices=cat_feature_indices,
        feature_set=feature_set,
        source_name=source_name,
        downsampled=downsampled,
    )

## 6. Model Evaluation Functions

In [ ]:
# =============================================================================
# MODEL EVALUATION FUNCTIONS
# =============================================================================

def evaluate_model_on_splits(
    model,
    X_splits: Dict[str, pd.DataFrame],
    y_splits: Dict[str, pd.Series],
    apply_scaling: bool = False,
    cat_feature_indices: Optional[List[int]] = None,
) -> Dict[str, Dict[str, float]]:
    """
    Train `model` on the train split, evaluate on val / test / oot / oot_strict.
    Returns a dict keyed by split name -> metrics dict.
    """
    model = clone(model)
    X_train, y_train = X_splits['train'], y_splits['train']

    scaler = None
    if apply_scaling:
        scaler = StandardScaler()
        X_train_proc = scaler.fit_transform(X_train)
    else:
        X_train_proc = X_train

    model_type = type(model).__name__
    t0 = time.time()

    if model_type == 'CatBoostClassifier':
        cat_idx = cat_feature_indices or []
        train_pool = Pool(X_train, y_train, cat_features=cat_idx)
        val_pool = Pool(X_splits['val'], y_splits['val'], cat_features=cat_idx)
        model.fit(train_pool, eval_set=val_pool, verbose=0)
    else:
        model.fit(X_train_proc, y_train)

    print(f"  Fit done ({model_type}): {time.time()-t0:.1f}s")

    results = {}
    for split_name in tqdm(['val', 'test', 'oot', 'oot_strict'], desc='Evaluating splits'):
        X_split = X_splits.get(split_name)
        y_split = y_splits.get(split_name)
        if X_split is None or len(X_split) == 0:
            continue
        if apply_scaling and scaler is not None:
            X_proc = scaler.transform(X_split)
        else:
            X_proc = X_split
        if model_type == 'CatBoostClassifier' and cat_feature_indices:
            pool = Pool(X_split, cat_features=cat_feature_indices)
            y_prob = model.predict_proba(pool)[:, 1]
        else:
            y_prob = model.predict_proba(X_proc)[:, 1]
        results[split_name] = compute_split_metrics(np.array(y_split), y_prob)
    return results


def evaluate_with_prepared_data(
    prepared_data: PreparedData,
    ml_model_object: Any,
    exp_name: str,
    apply_scaling: bool = False,
) -> Dict[str, Any]:
    """Evaluate one model using pre-prepared data. Returns flat metrics dict."""
    use_cat_features = (
        prepared_data.feature_set != 'embedding_only' and
        len(prepared_data.cat_feature_indices) > 0
    )
    split_results = evaluate_model_on_splits(
        model=ml_model_object,
        X_splits=prepared_data.X_splits,
        y_splits=prepared_data.y_splits,
        apply_scaling=apply_scaling,
        cat_feature_indices=prepared_data.cat_feature_indices if use_cat_features else None,
    )
    output = {
        'exp_name': exp_name,
        'model_type': type(ml_model_object).__name__,
        'source_name': prepared_data.source_name,
        'feature_set': prepared_data.feature_set,
        'n_features': len(prepared_data.feature_cols),
    }
    for split_name, metrics in split_results.items():
        for metric_name, value in metrics.items():
            output[f'{split_name}_{metric_name}'] = value
    return output


def get_prepared_cache_key(
    experiment_config: Dict,
    downsample_ratio: Optional[float] = None,
) -> tuple:
    """Resolve the prepared-data cache key for one experiment config."""
    dataset_name = experiment_config['dataset_name']
    feature_set = experiment_config.get('feature_set', 'embedding_only')
    experiment_downsample_ratio = experiment_config.get('downsample_ratio', downsample_ratio)
    return (dataset_name, feature_set, experiment_downsample_ratio)


def build_dataset_cache(
    experiment_configs: List[Dict],
    dataset_tables: Dict[str, str],
    verbose: bool = True,
) -> Dict[str, pd.DataFrame]:
    """Load each required modeling table once via Storage Read API."""
    dataset_cache: Dict[str, pd.DataFrame] = {}
    dataset_names_in_order = []
    for experiment_config in experiment_configs:
        dataset_name = experiment_config['dataset_name']
        if dataset_name not in dataset_names_in_order:
            dataset_names_in_order.append(dataset_name)

    for dataset_name in dataset_names_in_order:
        if dataset_name not in dataset_tables:
            raise KeyError(f"dataset_name not found in dataset_tables: {dataset_name}")
        print(f"Loading dataset cache: {dataset_name}")
        dataset_cache[dataset_name] = load_modeling_dataset(
            dataset_name,
            dataset_tables=dataset_tables,
            verbose=verbose,
        )
    return dataset_cache


def build_prepared_cache(
    experiment_configs: List[Dict],
    dataset_cache: Dict[str, pd.DataFrame],
    downsample_ratio: Optional[float] = None,
) -> Dict[tuple, PreparedData]:
    """Build PreparedData objects once per unique dataset/feature/downsample combination."""
    prepared_cache: Dict[tuple, PreparedData] = {}

    for experiment_config in experiment_configs:
        prepared_cache_key = get_prepared_cache_key(
            experiment_config,
            downsample_ratio=downsample_ratio,
        )
        if prepared_cache_key in prepared_cache:
            continue

        dataset_name, feature_set, experiment_downsample_ratio = prepared_cache_key
        print(
            f"Preparing cache: dataset={dataset_name}, "
            f"feature_set={feature_set}, downsample={experiment_downsample_ratio}"
        )
        prepared_cache[prepared_cache_key] = prepare_evaluation_data(
            df_features=dataset_cache[dataset_name],
            feature_set=feature_set,
            downsample_ratio=experiment_downsample_ratio,
            source_name=dataset_name,
        )

    return prepared_cache


def prepare_evaluation_caches(
    experiment_configs: List[Dict],
    dataset_tables: Dict[str, str],
    downsample_ratio: Optional[float] = None,
    verbose: bool = True,
) -> Tuple[Dict[str, pd.DataFrame], Dict[tuple, PreparedData]]:
    """Build dataset and prepared-data caches as a standalone preparation stage."""
    dataset_cache = build_dataset_cache(
        experiment_configs=experiment_configs,
        dataset_tables=dataset_tables,
        verbose=verbose,
    )
    prepared_cache = build_prepared_cache(
        experiment_configs=experiment_configs,
        dataset_cache=dataset_cache,
        downsample_ratio=downsample_ratio,
    )
    return dataset_cache, prepared_cache


def clear_evaluation_caches(
    dataset_cache: Optional[Dict[str, pd.DataFrame]] = None,
    prepared_cache: Optional[Dict[tuple, PreparedData]] = None,
) -> None:
    """Release dataset and prepared-data caches explicitly."""
    if prepared_cache is not None:
        prepared_cache.clear()
    if dataset_cache is not None:
        dataset_cache.clear()
    gc.collect()


def evaluate_single_experiment(
    experiment_config: Dict,
    prepared_cache: Dict[tuple, PreparedData],
    downsample_ratio: Optional[float] = None,
) -> Dict[str, Any]:
    """Run one experiment using a prebuilt prepared-data cache only."""
    prepared_cache_key = get_prepared_cache_key(
        experiment_config,
        downsample_ratio=downsample_ratio,
    )
    if prepared_cache_key not in prepared_cache:
        raise KeyError(
            "prepared_cache is missing key "
            f"{prepared_cache_key}. Build caches with prepare_evaluation_caches(...) first."
        )

    exp_name = experiment_config['exp_name']
    ml_model_object = experiment_config['ml_model_object']
    apply_scaling = experiment_config.get('apply_scaling', False)

    print(f"Running experiment: {exp_name}")
    return evaluate_with_prepared_data(
        prepared_data=prepared_cache[prepared_cache_key],
        ml_model_object=ml_model_object,
        exp_name=exp_name,
        apply_scaling=apply_scaling,
    )


def run_experiments_sequentially(
    experiment_configs: List[Dict],
    prepared_cache: Dict[tuple, PreparedData],
    downsample_ratio: Optional[float] = None,
) -> pd.DataFrame:
    """Run experiment configs one by one using prebuilt prepared-data caches."""
    results = []
    for experiment_idx, experiment_config in enumerate(experiment_configs, start=1):
        print(f"\n{'='*80}")
        print(
            f"Experiment {experiment_idx}/{len(experiment_configs)}: "
            f"{experiment_config['exp_name']}"
        )
        result = evaluate_single_experiment(
            experiment_config=experiment_config,
            prepared_cache=prepared_cache,
            downsample_ratio=downsample_ratio,
        )
        results.append(result)
    return pd.DataFrame(results)

## 7. SHAP Feature Importance Function

In [ ]:
# =============================================================================
# SHAP FEATURE IMPORTANCE (model-agnostic)
# Goal: quantify what proportion of top-N features are embeddings vs tabular.
# =============================================================================

def compute_shap_feature_importance(
    fitted_model,
    X_eval: pd.DataFrame,
    feature_cols: List[str],
    embedding_features: List[str],
    top_k_list: List[int] = [10, 20, 50],
    max_samples: int = 2000,
    random_state: int = 42,
    model_name: str = "",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute SHAP feature importance and embedding-vs-tabular breakdown.

    Returns:
        shap_summary_df : [feature, mean_abs_shap, rank, is_embedding]
        proportion_df   : [model_name, top_k, n_embedding_in_top_k,
                           proportion_embedding, n_tabular_in_top_k, proportion_tabular]
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"SHAP: {model_name or type(fitted_model).__name__}")
        print(f"{'='*60}")

    X_sample = X_eval
    if len(X_eval) > max_samples:
        X_sample = X_eval.sample(n=max_samples, random_state=random_state)
        if verbose:
            print(f"  Sampled {max_samples} of {len(X_eval)} rows")

    model_type = type(fitted_model).__name__
    if model_type == 'CatBoostClassifier':
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_sample)
    elif model_type in ('XGBClassifier', 'LGBMClassifier'):
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_sample)
    elif model_type == 'LogisticRegression':
        background = shap.sample(X_sample, min(100, len(X_sample)))
        explainer = shap.LinearExplainer(fitted_model, background)
        shap_values = explainer.shap_values(X_sample)
    else:
        background = shap.sample(X_sample, min(100, len(X_sample)))
        explainer = shap.KernelExplainer(fitted_model.predict_proba, background)
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    embedding_set = set(embedding_features)

    shap_summary_df = pd.DataFrame({
        'feature':       feature_cols,
        'mean_abs_shap': mean_abs_shap,
        'is_embedding':  [f in embedding_set for f in feature_cols],
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    shap_summary_df['rank'] = range(1, len(shap_summary_df) + 1)

    proportion_rows = []
    for k in top_k_list:
        k_actual = min(k, len(shap_summary_df))
        top_k_df = shap_summary_df.head(k_actual)
        n_emb = int(top_k_df['is_embedding'].sum())
        n_tab = k_actual - n_emb
        proportion_rows.append({
            'model_name':           model_name,
            'top_k':                k_actual,
            'n_embedding_in_top_k': n_emb,
            'proportion_embedding': n_emb / k_actual,
            'n_tabular_in_top_k':   n_tab,
            'proportion_tabular':   n_tab / k_actual,
        })

    proportion_df = pd.DataFrame(proportion_rows)
    if verbose:
        print("\nEmbedding proportion in top-K features:")
        print(proportion_df.to_string(index=False))
        print("\nTop 20 features by mean |SHAP|:")
        print(shap_summary_df.head(20)[['rank', 'feature', 'mean_abs_shap', 'is_embedding']].to_string(index=False))

    return shap_summary_df, proportion_df

## 8. Model Configurations

In [ ]:
# Standard CatBoost model
catboost_model = CatBoostClassifier(
    iterations=2500,
    depth=7,
    learning_rate=0.025,
    grow_policy='SymmetricTree',
    auto_class_weights='Balanced',
    od_wait=80,
    use_best_model=True,
    random_seed=42,
    verbose=0,
)

# Legacy-matched configuration (replicates best baseline hyperparameters)
catboost_model_legacy = CatBoostClassifier(
    iterations=2436,
    depth=7,
    learning_rate=0.027,          # rounded from 0.026766501358942353
    random_strength=3,
    l2_leaf_reg=2.95,
    border_count=136,
    min_data_in_leaf=30,
    grow_policy='SymmetricTree',
    od_wait=84,
    bootstrap_type='Bernoulli',
    subsample=0.79,
    leaf_estimation_iterations=8,
    loss_function='Logloss',
    eval_metric='AUC',
    od_type='Iter',
    use_best_model=True,
    random_seed=42,
    thread_count=-1,
    verbose=0,
)

## 9. Inspect Modeling Tables

In [ ]:
print("Lazy-loading mode enabled. Full modeling tables are streamed only when needed.")
print("Using BigQuery Storage Read API (parallel gRPC Arrow streams)")

for dataset_name, table_id in MODELING_TABLES.items():
    print("=" * 80)
    print(f"Dataset: {dataset_name}")
    print(f"Table: {table_id}")
    summary_df = query_modeling_table_summary(table_id)
    print(summary_df.to_string(index=False))

In [ ]:
# Quick dataset diagnostics without materializing full tables in memory
for dataset_name, table_id in MODELING_TABLES.items():
    print("\n" + "=" * 80)
    print(f"Digit distribution for {dataset_name}")
    digit_df = query_digit_distribution(table_id)
    display(digit_df)

    print(f"\nOutcome prevalence by split for {dataset_name}")
    prevalence_sql = f"""
    WITH base AS (
      SELECT
        CASE
          WHEN DATE(index_dt) > DATE('{FORMAL_EVAL_CUTOFF_DATE}') THEN 'oot'
          WHEN MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 10) BETWEEN 0 AND 7 THEN 'train'
          WHEN MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 10) = 8 THEN 'val'
          ELSE 'test'
        END AS split_name,
        outcome_flag
      FROM `{table_id}`
    )
    SELECT
      split_name,
      COUNT(*) AS n_rows,
      AVG(CAST(outcome_flag AS FLOAT64)) AS prevalence
    FROM base
    GROUP BY split_name
    ORDER BY CASE split_name WHEN 'train' THEN 1 WHEN 'val' THEN 2 WHEN 'test' THEN 3 ELSE 4 END
    """
    prevalence_df = client.query(prevalence_sql).to_dataframe()
    display(prevalence_df)

## 10. Experiment Configurations

In [ ]:
# Experiment configs split by dataset so we can load/evaluate/release one table
# at a time and avoid holding two 10M-row DataFrames in memory simultaneously.

new_te_experiment_configs = [
    {
        'dataset_name': 'new_te',
        'ml_model_object': catboost_model,
        'exp_name': 'exp_round10_exp2b_catboost_embedding_only',
        'feature_set': 'embedding_only',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'new_te',
        'ml_model_object': catboost_model,
        'exp_name': 'exp_round10_exp2b_catboost_tabular_only',
        'feature_set': 'tabular_only',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'new_te',
        'ml_model_object': catboost_model,
        'exp_name': 'exp_round10_exp2b_catboost_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'new_te',
        'ml_model_object': catboost_model_legacy,
        'exp_name': 'exp_round10_exp2b_catboost_legacy_tabular_only',
        'feature_set': 'tabular_only',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'new_te',
        'ml_model_object': catboost_model_legacy,
        'exp_name': 'exp_round10_exp2b_catboost_legacy_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
]

prod_rap_experiment_configs = [
    {
        'dataset_name': 'prod_rap',
        'ml_model_object': catboost_model,
        'exp_name': 'prod_rap_catboost_embedding_only',
        'feature_set': 'embedding_only',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'prod_rap',
        'ml_model_object': catboost_model,
        'exp_name': 'prod_rap_catboost_tabular_only',
        'feature_set': 'tabular_only',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'prod_rap',
        'ml_model_object': catboost_model,
        'exp_name': 'prod_rap_catboost_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
    {
        'dataset_name': 'prod_rap',
        'ml_model_object': catboost_model_legacy,
        'exp_name': 'prod_rap_catboost_legacy_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
]

## 11. New TE — Load, Evaluate, Release

In [ ]:
# Load, prepare, evaluate, and release NEW TE table only.
new_te_dataset_cache, new_te_prepared_cache = prepare_evaluation_caches(
    experiment_configs=new_te_experiment_configs,
    dataset_tables=MODELING_TABLES,
    downsample_ratio=GLOBAL_DOWNSAMPLE_RATIO,
    verbose=True,
)

new_te_results_df = run_experiments_sequentially(
    experiment_configs=new_te_experiment_configs,
    prepared_cache=new_te_prepared_cache,
    downsample_ratio=GLOBAL_DOWNSAMPLE_RATIO,
)
print(f"\nCompleted {len(new_te_results_df)} New TE experiments")
display(new_te_results_df)

# Release memory before loading the next table
clear_evaluation_caches(new_te_dataset_cache, new_te_prepared_cache)
del new_te_dataset_cache, new_te_prepared_cache
gc.collect()
print("New TE caches released.")

## 12. Prod RAP — Load, Evaluate, Release

In [ ]:
# Load, prepare, evaluate, and release PROD RAP table only.
prod_rap_dataset_cache, prod_rap_prepared_cache = prepare_evaluation_caches(
    experiment_configs=prod_rap_experiment_configs,
    dataset_tables=MODELING_TABLES,
    downsample_ratio=GLOBAL_DOWNSAMPLE_RATIO,
    verbose=True,
)

prod_rap_results_df = run_experiments_sequentially(
    experiment_configs=prod_rap_experiment_configs,
    prepared_cache=prod_rap_prepared_cache,
    downsample_ratio=GLOBAL_DOWNSAMPLE_RATIO,
)
print(f"\nCompleted {len(prod_rap_results_df)} Prod RAP experiments")
display(prod_rap_results_df)

# Release memory
clear_evaluation_caches(prod_rap_dataset_cache, prod_rap_prepared_cache)
del prod_rap_dataset_cache, prod_rap_prepared_cache
gc.collect()
print("Prod RAP caches released.")

## 13. Combined Results

In [ ]:
# Combine results from both tables
results_df = pd.concat([new_te_results_df, prod_rap_results_df], ignore_index=True)
print(f"Combined {len(results_df)} experiment results (New TE + Prod RAP)")
display(results_df)

## 13a. Results — Detailed Views

In [ ]:
# Full results transposed for readability
results_df.T

In [ ]:
# Key metrics summary — sorted by test AUC-ROC
key_cols = [
    'exp_name', 'source_name', 'feature_set', 'n_features',
    'test_auc_roc', 'test_lift_1pct', 'test_lift_5pct', 'test_lift_10pct',
    'test_precision_1pct', 'test_n_samples', 'test_n_positives', 'test_prevalence',
    'oot_auc_roc', 'oot_lift_1pct', 'oot_lift_5pct', 'oot_lift_10pct',
    'oot_precision_1pct', 'oot_n_samples', 'oot_n_positives', 'oot_prevalence',
]
available_cols = [column_name for column_name in key_cols if column_name in results_df.columns]
results_df[available_cols].sort_values('test_auc_roc', ascending=False)

In [ ]:
# OOT-strict results (digit-9 members only in post-cutoff period)
oot_strict_cols = [
    'exp_name', 'source_name', 'feature_set',
    'oot_strict_auc_roc', 'oot_strict_lift_1pct', 'oot_strict_lift_5pct',
    'oot_strict_precision_1pct', 'oot_strict_n_samples', 'oot_strict_n_positives',
]
available_oot_strict = [
    column_name for column_name in oot_strict_cols if column_name in results_df.columns
]
if available_oot_strict:
    results_df[available_oot_strict].sort_values('oot_strict_auc_roc', ascending=False)

## 14. SHAP Feature Importance — New TE Hybrid Model

In [ ]:
# Load only the new_te dataset needed for SHAP analysis
shap_dataset_name = 'new_te'
shap_feature_set = 'hybrid'
shap_model_template = catboost_model

print(f"Loading dataset for SHAP: {shap_dataset_name}")
df_shap_source = load_modeling_dataset(
    shap_dataset_name,
    dataset_tables=MODELING_TABLES,
    verbose=True,
)

prepared_new_te_shap = prepare_evaluation_data(
    df_features=df_shap_source,
    feature_set=shap_feature_set,
    downsample_ratio=GLOBAL_DOWNSAMPLE_RATIO,
    source_name=shap_dataset_name,
)

catboost_shap = clone(shap_model_template)
cat_features_new_te = prepared_new_te_shap.cat_feature_indices or []
train_pool_new_te = Pool(
    prepared_new_te_shap.X_splits['train'],
    prepared_new_te_shap.y_splits['train'],
    cat_features=cat_features_new_te,
)
val_pool_new_te = Pool(
    prepared_new_te_shap.X_splits['val'],
    prepared_new_te_shap.y_splits['val'],
    cat_features=cat_features_new_te,
)
catboost_shap.fit(train_pool_new_te, eval_set=val_pool_new_te, verbose=0)

print(
    f"Prepared SHAP data for {shap_dataset_name}: "
    f"train={prepared_new_te_shap.X_splits['train'].shape}, "
    f"test={prepared_new_te_shap.X_splits['test'].shape}"
)

del df_shap_source
gc.collect()

In [ ]:
# Embedding-vs-tabular proportion breakdown — New TE
commercial_proportion_df

In [ ]:
# Embedding-vs-tabular proportion breakdown
commercial_proportion_df

In [ ]:
# Top features by mean |SHAP|
commercial_shap_df.head(30)

## 14b. SHAP Feature Importance — Production RAP Hybrid Model

In [ ]:
# Load only the prod_rap dataset needed for SHAP analysis
prod_shap_dataset_name = 'prod_rap'
prod_shap_feature_set = 'hybrid'
prod_shap_model_template = catboost_model

print(f"Loading dataset for SHAP: {prod_shap_dataset_name}")
df_prod_shap_source = load_modeling_dataset(
    prod_shap_dataset_name,
    dataset_tables=MODELING_TABLES,
    verbose=True,
)

prepared_prod_shap = prepare_evaluation_data(
    df_features=df_prod_shap_source,
    feature_set=prod_shap_feature_set,
    downsample_ratio=GLOBAL_DOWNSAMPLE_RATIO,
    source_name=prod_shap_dataset_name,
)

catboost_shap_prod = clone(prod_shap_model_template)
cat_features_prod = prepared_prod_shap.cat_feature_indices or []
train_pool_prod = Pool(
    prepared_prod_shap.X_splits['train'],
    prepared_prod_shap.y_splits['train'],
    cat_features=cat_features_prod,
)
val_pool_prod = Pool(
    prepared_prod_shap.X_splits['val'],
    prepared_prod_shap.y_splits['val'],
    cat_features=cat_features_prod,
)
catboost_shap_prod.fit(train_pool_prod, eval_set=val_pool_prod, verbose=0)

print(
    f"Prepared SHAP data for {prod_shap_dataset_name}: "
    f"train={prepared_prod_shap.X_splits['train'].shape}, "
    f"test={prepared_prod_shap.X_splits['test'].shape}"
)

del df_prod_shap_source
gc.collect()

In [ ]:
# Embedding-vs-tabular proportion breakdown — Prod RAP
prod_proportion_df

In [ ]:
# Side-by-side embedding proportion comparison: New TE vs Production RAP
comparison_proportion = pd.concat([commercial_proportion_df, prod_proportion_df], ignore_index=True)
comparison_proportion

In [ ]:
# Top features by mean |SHAP| — Production RAP hybrid
prod_shap_df.head(30)

## 15. Save Results to Excel

In [ ]:
os.makedirs('experiment_logs', exist_ok=True)

output_path = (
    'experiment_logs/'
    'commercial_ip_formal_training_20241120_20250930_downstream_eval.xlsx'
)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Overall metrics
    results_df.T.to_excel(writer, sheet_name='all_metrics')

    # New TE SHAP
    commercial_shap_df.to_excel(writer, sheet_name='new_te_shap_summary', index=False)
    commercial_proportion_df.to_excel(writer, sheet_name='new_te_shap_proportion', index=False)

    # Production RAP SHAP
    prod_shap_df.to_excel(writer, sheet_name='prod_rap_shap_summary', index=False)
    prod_proportion_df.to_excel(writer, sheet_name='prod_rap_shap_proportion', index=False)

    # Combined embedding proportion comparison
    comparison_proportion.to_excel(writer, sheet_name='shap_proportion_comparison', index=False)

print(f"Saved → {output_path}")